# TPC-H data
Generates the TPC-H tables with DuckDB's tpch extension and writes them to the attached lakehouse as Delta
tables in the `tpch` schema (named `tpch_region` and so on in a lakehouse without schemas). Run it once, on this PC
or in Fabric; **TPC-H benchmark** reads them.

| scale_factor | order lines | size (about) |
|---|---|---|
| 0.1 | 600 thousand | 100 MB |
| 1 | 6 million | 1 GB |
| 10 | 60 million | 10 GB |
| 30 | 180 million | 30 GB |

In [ ]:
scale_factor = 1     # 0.1, 1, 10, 30 ...
schema = "tpch"      # where the tables go in the attached lakehouse
batch_rows = 2_000_000
TABLES = ['region', 'nation', 'supplier', 'customer', 'part', 'partsupp', 'orders', 'lineitem']

In [ ]:
import glob, os, subprocess, sys, tempfile, time
import duckdb
import pyarrow as pa
from fabric_desktop import lakehouse

def load_tpch(con):
    """DuckDB's tpch extension: from DuckDB's extension site, else the same extension from PyPI."""
    try:
        con.execute("INSTALL tpch")
        con.execute("LOAD tpch")
        return
    except Exception as e:
        print(f"DuckDB's extension site isn't reachable ({e}); installing the extension from PyPI")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"duckdb-extension-tpch=={duckdb.__version__}"])
    import duckdb_extension_tpch
    path = glob.glob(os.path.join(os.path.dirname(duckdb_extension_tpch.__file__), "extensions", "*", "tpch.duckdb_extension"))[0]
    con.execute(f"INSTALL '{path}'")
    con.execute("LOAD tpch")

# A database file rather than memory, so scale factors bigger than RAM still fit.
work = tempfile.mkdtemp(prefix="tpch_")
con = duckdb.connect(os.path.join(work, "tpch.duckdb"))
load_tpch(con)
started = time.perf_counter()
con.execute(f"CALL dbgen(sf={scale_factor})")
print(f"Generated scale factor {scale_factor} in {time.perf_counter() - started:.1f} s")

def table_name(schema, table):
    """schema.table, or schema_table in a lakehouse created without schemas."""
    return f"{schema}.{table}" if lakehouse.has_schemas() else f"{schema}_{table}"

In [ ]:
def slices(reader):
    """The query's rows in tables of about batch_rows, so a big table never has to fit in memory at once."""
    batches, pending = [], 0
    for batch in reader:
        batches.append(batch)
        pending += batch.num_rows
        if pending >= batch_rows:
            yield pa.Table.from_batches(batches)
            batches, pending = [], 0
    if batches:
        yield pa.Table.from_batches(batches)

for table in TABLES:
    started, rows, mode = time.perf_counter(), 0, "overwrite"
    result = con.execute(f"SELECT * FROM {table}")
    reader = (getattr(result, "to_arrow_reader", None) or result.fetch_record_batch)(batch_rows)  # newer, older DuckDB
    for part in slices(reader):
        lakehouse.write_table(part, table_name(schema, table), mode)
        rows, mode = rows + part.num_rows, "append"
    print(f"{table_name(schema, table)}: {rows:,} rows in {time.perf_counter() - started:.1f} s")
con.close()